In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

In [1]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
instance = service.active_account()["instance"]
backend_name = service.least_busy(operational=True, min_num_qubits=16).name

*Consulte a [referência da API](https://docs.quantum.ibm.com/api/functions/qunova-chemistry)*

> **Note:** As Qiskit Functions são um recurso experimental disponível apenas para usuários dos planos IBM Quantum&reg; Premium Plan, Flex Plan e On-Prem (via IBM Quantum Platform API) Plan. Elas estão em status de versão prévia e sujeitas a alterações.


<Accordion>
<AccordionItem title="Package versions">

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit-ibm-runtime~=0.45.0
```
</AccordionItem>
</Accordion>
## Visão geral
Na química quântica, o problema da estrutura eletrônica consiste em encontrar as soluções para a equação de Schrödinger eletrônica — as funções de onda quânticas que descrevem o comportamento dos elétrons do sistema. Essas funções de onda são vetores de amplitudes complexas, onde cada amplitude corresponde à contribuição de uma possível configuração eletrônica.

O estado fundamental é a função de onda de menor energia do sistema e tem uma importância especial no estudo de sistemas moleculares. A abordagem mais precisa para calcular o estado fundamental considera todas as configurações eletrônicas possíveis, mas isso se torna inviável para sistemas maiores, pois o número de configurações cresce exponencialmente com o tamanho do sistema.

O Handover Iterative Variational Quantum Eigensolver (HI-VQE) é um método híbrido quântico-clássico inovador para estimar com precisão o estado fundamental de sistemas moleculares. Ele integra hardware quântico com computação clássica, usando processadores quânticos para explorar eficientemente configurações eletrônicas candidatas e calculando a função de onda resultante em computadores clássicos. Ao gerar funções de onda compactas e quimicamente precisas, o HI-VQE aprimora a pesquisa e a descoberta em química quântica e ciência dos materiais.

![Imagem mostrando uma visão geral do algoritmo HI-VQE da Qunova](../docs/images/guides/qunova-chemistry/overview.svg)

O HI-VQE reduz a complexidade computacional do problema da estrutura eletrônica ao estimar o estado fundamental com alta precisão de forma eficiente. Ele se concentra em um subconjunto cuidadosamente selecionado das configurações eletrônicas mais relevantes, otimizando tanto a precisão quanto a eficiência.

Combinando os pontos fortes de computadores clássicos e quânticos, o HI-VQE refina e aprimora iterativamente a estimativa atual da função de onda. Suas técnicas únicas de construção de subespaço ajudam a tornar a seleção de configurações mais eficiente, para que os usuários tenham maior controle computacional e maior precisão nas simulações de química quântica.

Se você quiser conhecer o algoritmo com mais profundidade, pode [ler o artigo de pesquisa associado](https://arxiv.org/abs/2503.06292).

## Descrição
O número de configurações eletrônicas para um sistema molecular cresce exponencialmente com o tamanho do sistema. No entanto, para certos estados eletrônicos, como o estado fundamental, é comum que apenas uma pequena fração das configurações contribua de forma significativa para a energia do estado. Os métodos de interação de configuração selecionada (SCI) exploram essa esparsidade para reduzir os custos computacionais, identificando e focando nas configurações mais relevantes. Esse subconjunto de configurações é chamado de subespaço.

O HI-VQE aproveita a eficiência inerente dos computadores quânticos para representar sistemas moleculares e auxiliar na busca de subespaços. Ele integra sub-rotinas clássicas e quânticas para resolver o problema da estrutura eletrônica com alta precisão. Ao contrário dos métodos quânticos SCI existentes, o HI-VQE combina treinamento variacional, construção iterativa de subespaço e triagem de configurações por pré-diagonalização para aumentar a eficiência, reduzindo medições quânticas, iterações e custos de diagonalização clássica. O HI-VQE pode, portanto, ser aplicado a sistemas moleculares maiores que requerem mais qubits, e reduz o custo para resolver um problema de determinado tamanho com o mesmo grau de precisão.

![Imagem mostrando uma descrição detalhada de cada etapa do algoritmo HI-VQE da Qunova.](../docs/images/guides/qunova-chemistry/description.avif)

Para calcular o estado fundamental de um sistema, o HI-VQE usa primeiro o pacote de química clássica PySCF para gerar uma representação molecular a partir das entradas fornecidas pelo usuário, como a geometria molecular e outras informações moleculares. Em seguida, entra em um loop de otimização híbrido quântico-clássico, refinando iterativamente um subespaço para representar de forma otimizada o estado fundamental enquanto minimiza o número de configurações incluídas. O loop continua até que critérios de convergência, como tamanho do subespaço ou estabilidade de energia, sejam satisfeitos, após o que a função de onda do estado fundamental calculada e sua energia são produzidas como saída. Esses resultados podem ser usados para construir superfícies de energia potencial precisas e realizar análises mais aprofundadas do sistema.

O loop de otimização se concentra em ajustar os parâmetros de um circuito quântico para gerar um subespaço de alta qualidade. O HI-VQE oferece três opções de circuito quântico: [`excitation_preserving`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.excitation_preserving), [efficient_su2](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.efficient_su2) e [LUCJ](https://qiskit-community.github.io/ffsim/explanations/lucj.html). A otimização é inicializada próximo ao estado de referência de Hartree-Fock devido à sua adequação geral. O circuito é então executado em um dispositivo quântico e as configurações são amostradas do estado quântico resultante antes de serem retornadas como strings binárias. Devido ao ruído do dispositivo quântico, algumas configurações amostradas podem ser fisicamente inválidas, não conservando o número de elétrons ou o spin. O HI-VQE resolve isso usando o processo de recuperação de configuração do pacote [qiskit-addon-sqd](/guides/qiskit-addons-sqd#sample-based-quantum-diagonalization-sqd-overview), para que os usuários possam corrigir configurações inválidas ou descartá-las.

As configurações válidas passam então por uma etapa de triagem opcional para remover aquelas que se prevê contribuir minimamente. Isso reduz a dimensão do subespaço, diminuindo assim o custo da etapa de diagonalização. Se a triagem estiver habilitada, um Hamiltoniano de subespaço preliminar é construído a partir das configurações válidas e uma diagonalização é realizada com critérios de encerramento muito relaxados. Embora a precisão das amplitudes resultantes para cada configuração seja baixa, ela é eficaz para prever quais configurações deixar de fora do subespaço nesta iteração, e é rápida de calcular.

As configurações selecionadas são adicionadas ao subespaço e o Hamiltoniano do sistema é projetado nesse subespaço. O subespaço se atualiza iterativamente, preservando as configurações mais relevantes entre as iterações. Essa abordagem contrasta com métodos alternativos porque o circuito quântico não precisa aproximar o estado fundamental completo a cada etapa.

Em seguida, o Hamiltoniano do subespaço é diagonalizado classicamente para obter o menor autovalor e seu autovetor correspondente, representando uma aproximação do estado fundamental e sua energia. À medida que a qualidade do subespaço melhora com as iterações, o estado fundamental calculado se aproxima melhor do estado fundamental verdadeiro. Uma etapa adicional de triagem pode ser realizada neste ponto para remover quaisquer configurações do subespaço que não tenham uma contribuição substancial para o estado fundamental calculado. Essa etapa garante que o subespaço levado para a próxima iteração seja o mais compacto possível. Isso é avaliado com base nas amplitudes retornadas pela diagonalização, pois elas representam a contribuição de importância de cada configuração para o estado fundamental calculado.

Uma verificação de convergência então determina se um treinamento adicional melhoraria os resultados. Se sim, uma etapa de expansão clássica opcional é realizada, os parâmetros do circuito quântico são atualizados para minimizar ainda mais a energia calculada e o processo se repete. A etapa de expansão clássica gera configurações adicionais para o subespaço, complementando as configurações amostradas do dispositivo quântico. Ela primeiro identifica a configuração com a maior amplitude nos resultados da diagonalização, antes de gerar novas configurações com excitações simples e duplas a partir da configuração identificada. O número desejado dessas configurações é então adicionado ao subespaço.

Uma vez determinado que as iterações convergiram, o HI-VQE retorna o estado fundamental calculado (na forma dos estados no subespaço e suas amplitudes na função de onda do estado fundamental), sua energia e uma medida de variância de energia que indica se o estado calculado forma um autoestado do Hamiltoniano do sistema.

Os usuários podem decidir o circuito quântico utilizado e o número de shots tomados para cada circuito quântico, bem como controlar o tamanho do subespaço ou habilitar a geração clássica de configurações adicionais para auxiliar as configurações geradas pelo quantum. Dessa forma, os usuários podem adaptar o comportamento do HI-VQE para suas aplicações desejadas.

## Licenciamento
Observe que o uso desta Qiskit Function é limitado a problemas que requerem no máximo 20 qubits, a menos que seja obtida uma licença que conceda um limite maior.

Envie um e-mail para [qiskit.support@qunovacomputing.com](mailto:qiskit.support@qunovacomputing.com) se quiser solicitar informações sobre como obter uma licença.

## Primeiros passos
Primeiro, [solicite acesso à função](https://forms.office.com/r/zN3hvMTqJ1).
Em seguida, autentique-se usando sua [chave de API do IBM Quantum&reg;](http://quantum.cloud.ibm.com/) e, assumindo que você já [salvou sua conta](/guides/functions#install-qiskit-functions-catalog-client) no seu ambiente local, selecione a Qiskit Function da seguinte forma:

In [1]:
import reprlib
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Verify that you have access to the function
catalog.list()

[QiskitFunction(qunova/hivqe-chemistry),
 QiskitFunction(global-data-quantum/quantum-portfolio-optimizer),
 QiskitFunction(algorithmiq/tem),
 QiskitFunction(qedma/qesem),
 QiskitFunction(multiverse/singularity),
 QiskitFunction(ibm/circuit-function),
 QiskitFunction(q-ctrl/optimization-solver),
 QiskitFunction(colibritd/quick-pde),
 QiskitFunction(q-ctrl/performance-management),
 QiskitFunction(kipu-quantum/iskay-quantum-optimizer)]

In [ ]:
# Load the function
function = catalog.load("qunova/hivqe-chemistry")

Opções adicionais podem ser definidas e fornecidas para o sistema molecular no seguinte formato de dicionário.

In [3]:
# Define the molecule geometry
geometry = """
N         -0.85188       -0.02741        0.03141;
H          0.16545        0.00593       -0.01648;
H         -1.16348       -0.39357       -0.86702;
H         -1.16348        0.94228        0.06281;
"""

Execute a função com as entradas de geometria e opções.

In [4]:
# Configure some options for the job.
molecule_options = {"basis": "sto3g"}
hivqe_options = {"shots": 100, "max_iter": 20}

É uma boa ideia imprimir o ID do job da Function para que ele possa ser fornecido em solicitações de suporte caso algo dê errado.

In [ ]:
# Run HI-VQE
job = function.run(
    geometry=geometry,
    # `backend_name` is the name of a backend with at least 16 qubits,
    # for example, "ibm_marrakesh".
    backend_name=backend_name,
    max_states=2000,
    max_expansion_states=10,
    molecule_options=molecule_options,
    hivqe_options=hivqe_options,
)

It is a good idea to print the Function job ID so that it can be provided in support requests if something goes wrong.

In [6]:
print("Job ID:", job.job_id)

Job ID: e5ced6f2-fd1d-4244-a6aa-bd27cfb0cdee


Este exemplo usa então 16 qubits com 8 orbitais da base sto3g para uma molécula de NH3.
Verifique o [status](/guides/functions#check-job-status) do workload da sua Qiskit Function ou obtenha os [resultados](/guides/functions#retrieve-results) da seguinte forma:

In [7]:
print(job.status())

QUEUED


After the job is completed, the results can be obtained with `result()` instance.

In [8]:
result = job.result()

# Output can be long, so we display a shortened representation
shortened_result = reprlib.repr(result)
print(shortened_result)

{'eigenvector': [0.9824448589364075, 0.009527106392132133, 6.854074372058527e-08, 3.591500190038039e-07, 0.0012975231577544268, 2.310159709002111e-05, ...], 'energy': -55.52108557170985, 'energy_history': [-55.51901898989887, -55.52056881448526, -55.52065046778772, -55.520690696813716, -55.520691108428, -55.520708448092634, ...], 'energy_variance': 3.066239097617371e-10, ...}


Após a conclusão do job, os resultados podem ser obtidos com a instância `result()`.

In [ ]:
fci_energy = -55.521148034704126  # the exact energy using FCI method
hivqe_energy = result["energy"]
print(
    f"|Exact Energy - HI-VQE Energy|: "
    f"{abs(fci_energy - hivqe_energy) * 1000} mHa"
)
print(f"Sampled Number of States: {len(result['states'])}")

|Exact Energy - HI-VQE Energy|: 0.06246299427914437 mHa
Sampled Number of States: 1936


## Performance

This section shows the demonstrated benchmark calculations of HI-VQE with a 24-qubit case for Li2S, a 40-qubit case for an N2 molecule, and a 44-qubit case for an FeP-NO system.

#### Dissociation potential energy surface curve for an Li2S molecule with 24 qubits

The PES curve is shown with the FCI reference and initial guess from RHF, together with the energy error from the FCI reference.

![Image showing that HI-VQE produces solutions within chemical accuracy of a classical reference PES curve for the Li2S system](../docs/images/guides/qunova-chemistry/Li2S_PES.avif).

The calculations have been conducted with the following geometries and options.

In [10]:
# This cell is hidden from users
backend_name = service.least_busy(operational=True, min_num_qubits=38).name

In [11]:
# Define Li2S geometries
Li2S_geoms = {
    "Li2S_1.51": "S        -1.239044    0.671232   -0.030374;Li       -1.506327    0.432403   -1.498949;Li       -0.899996    0.973348    1.826768;",
    "Li2S_2.40": "S        -1.741432    0.680397    0.346702;Li       -0.529307    0.488006   -1.729343;Li       -1.284307    0.989409    2.177209;",
    "Li2S_3.80": "S        -2.707255    0.674298    0.909161;Li        0.079218    0.552012   -1.671656;Li       -0.927010    0.931502    1.557063;",
}

# Configure some options for the job.
molecule_options = {
    "basis": "sto3g",
}
hivqe_options = {
    "shots": 100,
    "max_iter": 20,
}

results = []
for geom in ["Li2S_1.51", "Li2S_2.40", "Li2S_3.80"]:
    # Run HI-VQE
    job = function.run(
        geometry=Li2S_geoms[geom],
        backend_name=backend_name,  # can use any device with at least 38 qubits
        max_states=2000,
        max_expansion_states=10,
        molecule_options=molecule_options,
        hivqe_options=hivqe_options,
    )
    results.append(job.result())

The red dots represent the HI-VQE calculation results for six different geometries, and three geometries corresponding to 1.51, 2.40, and 3.80 Angstrom are provided as input in the above cell.

#### Dissociation PES curve for an N2 molecule with 40 qubits

The nitrogen molecule has been identified as a multireference system with large correlation energy contributions beyond the Hartree-Fock state. We conducted a benchmark calculation for the N2 molecule with cc-pvdz basis, (20o,14e) using the homo-lumo active orbital selection. The complete active space (CAS) number to represent this problem is 6,009,350,400. It is not possible to obtain the eigenvalue problem solution (for energy and electronic structure) with this number of states using a powerful desktop (16cpu/64GB). With HI-VQE, users can efficiently search the subspace of CAS states to find chemically accurate results while saving computation resources significantly. The following plots show the PES curve of 40 qubits HI-VQE calculation of the N2 molecule dissociation.

![Image showing that HI-VQE produces solutions within chemical accuracy of a classical reference PES curve for the N2 system](../docs/images/guides/qunova-chemistry/N2_PES_40qubits.avif)

#### Dissociation PES curve for five-coordinated iron(II)-porphyrin with an NO system with 44 qubits

Another interesting chemical system is an iron(II)-porphyrin (FeP) complex with a coordinated nitric oxide (NO) ligand, which represents a biologically relevant metalloporphyrin system that plays crucial roles in various physiological processes. In this example, HI-VQE has been utilized to estimate the accurate potential energy surface curve of the intermolecular interaction between FeP and NO (ground state energy for differently separated geometries). The combined system has 450 orbitals and 202 electrons (450o,202e) with 6-31g(d) basis in total. The homo-lumo active orbital selection was utilized to calculate the smaller case from the real case with (22o,22e). From the following benchmark results, we were able to achieve the chemical accuracy (>&nbsp;1.6 mHa) with a state-of-the-art classical computer chemistry calculation of CASCI(DMRG) (22o,22e) reference.

![Image showing that HI-VQE produces solutions within chemical accuracy of a classical reference PES curve for the FeP-NO system](../docs/images/guides/qunova-chemistry/fepno_44qubits.avif)

## Benchmarks

- Exact matrix size is the number of determinants for exact solution, such as FCI and CASCI.
- HI-VQE calculation samples and calculates the subspace of it (as in, HI-VQE matrix size).
- Total time includes QPU runtime and Qiskit Function runs with CPU.
- Accuracy is estimated from the energy difference from exact solution.

|   Chemical system  | Number of qubits | Exact matrix size |  HI-VQE matrix size  | E(diff) from exact (mHa) | Number of iteration | Total time    | QPU runtime usage |
| ------------------ | ---------------- | ----------------- | ------------------- | ------------------------ | ------------------- | ------------- | ----------------- |
|  $NH_3$ (8o,10e)   |  16              |  3136             |  1936               |  0.08                    |  6                  | 37 s          | 34 s              |
|  $Li_2S$ (10o,10e) |  20              |  63504            |  3969               |  0.60                    |  5                  | 250 s         | 50 s              |
|  $NH_3$ (15o,10e)  |  30              |  9018009          |  49729              |  0.90                    |  5                  | 354 s         | 54 s              |
|  $N_2$ (16o,14e)   |  32              |  130873600        |  1798281            |  1.10                    |  9                  | 6531 s        | 121 s             |
|  $3H_2O$ (18o,24e) |  36              |  344622096        |  399424             |  0.90                    |  24                 | 5174 s        | 130 s             |
|  $N_2$ (20o,14e)   |  40              |  6009350400       |  9012004            |  1.20                    |  21                 | 46547 s       | 258 s             |

## Fetch error messages

If your workload fails, the status will be `ERROR` and calling `job.result()` will raise an exception:

In [12]:
job = function.run(
    geometry="invalid-geometry",  # This will cause an error
    backend_name=backend_name,
    max_states=2000,
    max_expansion_states=15,
    molecule_options=molecule_options,
    hivqe_options=hivqe_options,
)

job.result()

QiskitServerlessException: ["runner.UnknownRuntimeError: 'An unexpected error occurred during job execution. Please make sure that your inputs are valid. If you are still experiencing problems, you can contact the Qunova Computing support service at qiskit.support@qunovacomputing.com and provide the Function job ID of this job for more assistance. -- https://docs.quantum.ibm.com/errors#1500'\n"]

In [13]:
job.status()

'ERROR'

## Get support

You can send an email to [qiskit.support@qunovacomputing.com](mailto:qiskit.support@qunovacomputing.com) for help with this function.

If you want help with troubleshooting a specific error, please provide the Function job ID of the job that encountered the error.

## Next steps

<Admonition type="tip" title="Recommendations">
- Request access to the function by filling in [this form](https://forms.office.com/r/zN3hvMTqJ1).
- Visit the [API reference](/docs/api/functions/qunova-chemistry) for this Qiskit Function.
- Try the [Compute dissociation PES curve for FeP-NO with HI-VQE](/docs/tutorials/qunova-hivqe) tutorial.
- Review [Pellow-Jarman, A., et al. (2025).  HIVQE: Handover Iterative Variational Quantum Eigensolver for Efficient Quantum Chemistry Calculations. arXiv preprint arXiv:2503.06292](https://arxiv.org/abs/2503.06292).
- Try the [Dissociation PES curves with Qunova HiVQE](/docs/tutorials/qunova-hivqe) tutorial.
</Admonition>